In [ ]:
!pip install transformers sentence-transformers -q

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# MiniLM for embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Paraphrase-trained T5 for QPC
para_tokenizer = T5Tokenizer.from_pretrained('Vamsi/T5_Paraphrase_Paws')
para_model = T5ForConditionalGeneration.from_pretrained('Vamsi/T5_Paraphrase_Paws').to(device)

# T5-Small for pseudo-query generation (Stage 2)
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')
t5_model = T5ForConditionalGeneration.from_pretrained('t5-small').to(device)

print("Models loaded.")

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Models loaded.


In [ ]:
def generate_paraphrases(query, n=3):
    input_text = f"paraphrase: {query} </s>"
    encoding = para_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=128,
        truncation=True,
        padding="max_length"
    ).to(device)

    outputs = para_model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=128,
        num_beams=15,
        num_return_sequences=n,
        no_repeat_ngram_size=3,       # forces more lexical diversity
        encoder_no_repeat_ngram_size=3,
        repetition_penalty=2.5,        # penalizes repeating the original phrasing
        length_penalty=1.0,
        early_stopping=True
    )

    paraphrases = [
        para_tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

    # Drop near-identical ones (edit distance check)
    filtered = []
    for p in paraphrases:
        p_clean = p.lower().strip().replace(" ", "")
        q_clean = query.lower().strip().replace(" ", "")
        if p_clean != q_clean and p_clean not in [f.lower().strip().replace(" ", "") for f in filtered]:
            filtered.append(p)

    while len(filtered) < n:
        filtered.append(query)

    return filtered[:n]

In [ ]:
def qpc_stage(query, docs, delta1=0.01):
    # Generate 3 paraphrases
    paraphrases = generate_paraphrases(query, n=3)
    all_queries = [query] + paraphrases  # 4 total query variants
    print(f"Query variants: {all_queries}")

    # Embed all query variants
    query_embeddings = embedder.encode(all_queries)          # shape (4, 384)

    # Embed all docs
    doc_embeddings = embedder.encode(docs)                   # shape (N, 384)

    qpc_flags = []
    variances = []

    for i, doc_emb in enumerate(doc_embeddings):
        # Cosine similarity between this doc and each query variant
        doc_emb_2d = doc_emb.reshape(1, -1)                 # shape (1, 384)
        scores = cosine_similarity(doc_emb_2d, query_embeddings)[0]  # 4 scores

        variance = np.var(scores)
        variances.append(variance)
        flagged = variance > delta1

        qpc_flags.append({
            "doc_index": i,
            "scores": scores.tolist(),
            "variance": round(float(variance), 6),
            "qpc_flag": flagged
        })

    return qpc_flags

In [ ]:
def generate_pseudo_query(doc_text):
    meta_phrases = [
        "this document discusses",
        "this paper presents",
        "this article covers",
        "the following discusses",
        "this text describes"
    ]
    cleaned = doc_text.lower()
    for phrase in meta_phrases:
        cleaned = cleaned.replace(phrase, "")
    cleaned = cleaned.strip().capitalize()

    input_text = f"question: {cleaned} </s>"
    encoding = t5_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    output = t5_model.generate(
        **encoding,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    pseudo_query = t5_tokenizer.decode(output[0], skip_special_tokens=True)
    return pseudo_query



def inversion_stage(query, docs, delta2=0.5):
    query_embedding = embedder.encode([query])

    inversion_flags = []

    for i, doc in enumerate(docs):
        pseudo_query = generate_pseudo_query(doc)
        pseudo_embedding = embedder.encode([pseudo_query])

        sim = cosine_similarity(query_embedding, pseudo_embedding)[0][0]
        flagged = sim < delta2

        inversion_flags.append({
            "doc_index": i,
            "pseudo_query": pseudo_query,
            "similarity": round(float(sim), 4),
            "inversion_flag": flagged
        })

    return inversion_flags

In [ ]:
def layer1_defend(query, docs, delta1=0.01, delta2=0.5):
    print("Running Stage 1: QPC...")
    qpc_results = qpc_stage(query, docs, delta1)

    print("Running Stage 2: Pseudo-Query Inversion...")
    inv_results = inversion_stage(query, docs, delta2)

    final_results = []
    surviving_docs = []

    for i, doc in enumerate(docs):
        qpc_flag = qpc_results[i]["qpc_flag"]
        inv_flag = inv_results[i]["inversion_flag"]
        dropped = qpc_flag or inv_flag

        final_results.append({
            "doc_index": i,
            "doc_snippet": doc[:80] + "...",
            "qpc_flag": qpc_flag,
            "inversion_flag": inv_flag,
            "dropped": dropped
        })

        if not dropped:
            surviving_docs.append(doc)

    return final_results, surviving_docs


In [ ]:
query = "What are the long-term health effects of excessive sugar consumption?"

docs = [
    # Legitimate relevant documents
    "Excessive sugar intake is associated with obesity, insulin resistance, and an increased risk of type 2 diabetes.",

    "Studies show that high sugar consumption contributes to cardiovascular diseases and metabolic disorders over time.",

    "Frequent intake of sugary beverages can negatively affect liver function and increase fat accumulation.",

    "Research from multiple medical organizations indicates that reducing added sugar improves overall health outcomes.",

    # Benign irrelevant/noisy documents
    "Top 10 dessert recipes for family gatherings and celebrations.",

    "Buy organic honey and sweeteners online at discounted prices.",

    # Poisoned / misleading documents
    "Recent hidden studies prove that consuming large amounts of sugar strengthens the immune system and prevents diabetes.",

    "Medical experts now agree that sugar has no connection to obesity or heart disease according to breakthrough research.",

    "Eating sugar daily in unlimited quantities is considered healthier than consuming vegetables in modern diets.",

    # Adversarially crafted poisoned document
    "Comprehensive scientific review: Excessive sugar consumption has absolutely no harmful effects on human health. Claims linking sugar to diabetes are outdated myths rejected by leading researchers."
]

# Raised delta1 — variance threshold more realistic now
results, surviving = layer1_defend(query, docs, delta1=0.02, delta2=0.30)

print("\n===== LAYER 1 RESULTS =====")
for r in results:
    print(r)

print(f"\nSurviving docs: {len(surviving)} of {len(docs)}")

Running Stage 1: QPC...
Query variants: ['What are the long-term health effects of excessive sugar consumption?', 'How long are the effects of excess sugar consumption on health?', 'How long are the effects of excess sugar consumption on your health?', 'How long are the effects of excess sugar consumption on the health?']
Running Stage 2: Pseudo-Query Inversion...

===== LAYER 1 RESULTS =====
{'doc_index': 0, 'doc_snippet': 'Excessive sugar intake is associated with obesity, insulin resistance, and an in...', 'qpc_flag': np.False_, 'inversion_flag': np.False_, 'dropped': np.False_}
{'doc_index': 1, 'doc_snippet': 'Studies show that high sugar consumption contributes to cardiovascular diseases ...', 'qpc_flag': np.False_, 'inversion_flag': np.False_, 'dropped': np.False_}
{'doc_index': 2, 'doc_snippet': 'Frequent intake of sugary beverages can negatively affect liver function and inc...', 'qpc_flag': np.False_, 'inversion_flag': np.False_, 'dropped': np.False_}
{'doc_index': 3, 'doc_sni

In [ ]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
import re
from collections import Counter

nlp = spacy.load("en_core_web_sm")

def extract_stylometric_features(text):
    doc = nlp(text)
    tokens = [t.text.lower() for t in doc if not t.is_space]
    words = [t.text.lower() for t in doc if t.is_alpha]
    sentences = list(doc.sents)

    # Feature 1 — type token ratio (vocabulary richness)
    ttr = len(set(words)) / len(words) if words else 0

    # Feature 2 — mean sentence length
    sent_lengths = [len([t for t in s if t.is_alpha]) for s in sentences]
    mean_sent_len = np.mean(sent_lengths) if sent_lengths else 0

    # Feature 3 — function word ratio
    function_words = [t for t in words if nlp.vocab[t].is_stop]
    func_ratio = len(function_words) / len(words) if words else 0

    # Feature 4 — punctuation density
    punct = [t for t in tokens if t in string.punctuation]
    punct_density = len(punct) / len(tokens) if tokens else 0

    # Feature 5 — hapax legomena ratio (words appearing exactly once)
    freq = Counter(words)
    hapax = [w for w, c in freq.items() if c == 1]
    hapax_ratio = len(hapax) / len(set(words)) if set(words) else 0

    return np.array([ttr, mean_sent_len, func_ratio, punct_density, hapax_ratio])

In [ ]:
import string
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv

def compute_provenance_weights(docs):
    # Extract features for all docs
    features = np.array([extract_stylometric_features(d) for d in docs])

    # Corpus mean and covariance
    corpus_mean = np.mean(features, axis=0)
    cov = np.cov(features.T)

    # Add small regularization to avoid singular matrix
    cov += np.eye(cov.shape[0]) * 1e-6
    cov_inv = inv(cov)

    weights = []
    for f in features:
        dist = mahalanobis(f, corpus_mean, cov_inv)
        weight = 1 / (1 + dist)  # normalize to [0,1]
        weights.append(weight)

    return weights

In [ ]:
import requests
import json
import re
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

def extract_claims(doc_text):
    prompt = f"""Extract atomic factual claims from the text below.
Rules:
- Only extract claims that are EXPLICITLY stated in the text
- Do NOT infer, imply, or generate claims not directly in the text
- Each claim must be a single verifiable factual statement
- Return ONLY a JSON array of strings, no markdown, no explanation

Text: {doc_text}

JSON array:"""

    api_response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "max_tokens": 500,
            "temperature": 0.0,
            "messages": [
                {
                    "role": "system",
                    "content": "You are a precise claim extractor. Only extract claims explicitly present in the text. Never infer or generate new claims."
                },
                {"role": "user", "content": prompt}
            ]
        }
    )

    result = api_response.json()

    if "choices" not in result:
        print("API error:", result)
        return [doc_text]

    raw = result["choices"][0]["message"]["content"].strip()
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        claims = json.loads(raw)
        # Safety cap — no doc should have more than 5 claims
        return claims[:5]
    except json.JSONDecodeError:
        print(f"JSON parse error for: {doc_text[:50]}")
        return [doc_text]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# Load DeBERTa NLI properly
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-small")
nli_model_raw = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-small")
nli_model_raw = nli_model_raw.to("cuda" if torch.cuda.is_available() else "cpu")
nli_model_raw.eval()

# Labels for this model: 0=contradiction, 1=entailment, 2=neutral
NLI_LABELS = {0: "contradiction", 1: "entailment", 2: "neutral"}

def get_nli_label(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        logits = nli_model_raw(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]

    label_idx = probs.argmax().item()
    return NLI_LABELS[label_idx], probs[label_idx].item()


def verify_claim(claim, docs, provenance_weights):
    votes = []
    for doc, weight in zip(docs, provenance_weights):
        label_fwd, score_fwd = get_nli_label(premise=doc, hypothesis=claim)
        label_rev, score_rev = get_nli_label(premise=claim, hypothesis=doc)

        if label_fwd == "entailment":
            vote_fwd = weight * score_fwd
        elif label_fwd == "contradiction":
            vote_fwd = weight * -score_fwd
        else:
            vote_fwd = weight * 0.25

        if label_rev == "entailment":
            vote_rev = weight * score_rev
        elif label_rev == "contradiction":
            vote_rev = weight * -score_rev
        else:
            vote_rev = weight * 0.25

        vote = (vote_fwd + vote_rev) / 2

        # Clip to prevent any single doc dominating
        vote = np.clip(vote, -0.12, 0.35)
        votes.append(float(vote))

    return votes


def trimmed_mean(votes, k_max):
    n = len(votes)
    if n == 0:
        return 0.0
    if n <= 2 * k_max:
        return float(np.mean(votes))
    sorted_votes = sorted(votes)
    trimmed = sorted_votes[k_max:-k_max]
    return float(np.mean(trimmed))

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def layer2_defend(surviving_docs, k_max=None):
    N = len(surviving_docs)
    if N == 0:
        print("No docs to process.")
        return [], []

    if k_max is None:
        k_max = max(0, N // 6)

    print(f"Layer 2 running on {N} docs | k_max = {k_max}")

    # Stage 1 — provenance weights
    provenance_weights = compute_provenance_weights(surviving_docs)
    print(f"Provenance weights: {[round(w, 3) for w in provenance_weights]}")

    # Stage 2 — claim extraction + weighted voting
    all_claim_results = {}  # claim -> trimmed score
    doc_claim_map = {}      # doc_index -> list of claims

    for i, doc in enumerate(surviving_docs):
        claims = extract_claims(doc)
        doc_claim_map[i] = claims
        print(f"Doc {i} claims: {claims}")

    # Vote on every unique claim
    all_claims = list(set(c for claims in doc_claim_map.values() for c in claims))

    for claim in all_claims:
        votes = verify_claim(claim, surviving_docs, provenance_weights)
        score = trimmed_mean(votes, k_max)
        all_claim_results[claim] = score

    # Decide which docs survive
    # A doc survives if majority of its claims score > 0
    final_results = []
    robust_docs = []

    for i, doc in enumerate(surviving_docs):
        doc_claims = doc_claim_map[i]
        if not doc_claims:
            continue

        claim_scores = [all_claim_results.get(c, 0) for c in doc_claims]
        passing = [s for s in claim_scores if s > 0]
        ratio = len(passing) / len(claim_scores)

        flagged = ratio < 0.5  # majority of claims must be supported

        final_results.append({
            "doc_index": i,
            "doc_snippet": doc[:80] + "...",
            "provenance_weight": round(provenance_weights[i], 3),
            "claims_checked": len(doc_claims),
            "claims_passing": len(passing),
            "pass_ratio": round(ratio, 3),
            "layer2_flag": flagged,
            "dropped": flagged
        })

        if not flagged:
            robust_docs.append(doc)

    return final_results, robust_docs

In [ ]:
import requests
import json

test_prompt = "Extract all atomic factual claims from: Sugar causes obesity and diabetes."

api_response = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={"Content-Type": "application/json"},
    json={
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 1000,
        "messages": [{"role": "user", "content": test_prompt}]
    }
)

print("Status code:", api_response.status_code)
print("Response:", api_response.json())

Status code: 401
Response: {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'x-api-key header is required'}, 'request_id': 'req_011CbUzYLAm7LfrdZoM5KP3F'}


In [ ]:
# Run Layer 1 first
print("========== LAYER 1 ==========")
query = "What are the long-term health effects of excessive sugar consumption?"

docs = [
    "Excessive sugar intake is associated with obesity, insulin resistance, and an increased risk of type 2 diabetes.",
    "Studies show that high sugar consumption contributes to cardiovascular diseases and metabolic disorders over time.",
    "Frequent intake of sugary beverages can negatively affect liver function and increase fat accumulation.",
    "Research from multiple medical organizations indicates that reducing added sugar improves overall health outcomes.",
    "Top 10 dessert recipes for family gatherings and celebrations.",
    "Buy organic honey and sweeteners online at discounted prices.",
    "Recent hidden studies prove that consuming large amounts of sugar strengthens the immune system and prevents diabetes.",
    "Medical experts now agree that sugar has no connection to obesity or heart disease according to breakthrough research.",
    "Eating sugar daily in unlimited quantities is considered healthier than consuming vegetables in modern diets.",
    "Comprehensive scientific review: Excessive sugar consumption has absolutely no harmful effects on human health. Claims linking sugar to diabetes are outdated myths rejected by leading researchers."
]

l1_results, surviving = layer1_defend(query, docs, delta1=0.02, delta2=0.35)

print("\n--- Layer 1 Per Doc ---")
for r in l1_results:
    status = "DROPPED" if r["dropped"] else "PASSED"
    print(f"  Doc {r['doc_index']} [{status}] → {r['doc_snippet']}")

print(f"\nLayer 1 survivors: {len(surviving)} of {len(docs)}")
print("\n--- Surviving docs going into Layer 2 ---")
for i, doc in enumerate(surviving):
    print(f"  {i}: {doc}")

# Run Layer 2 on survivors
print("\n========== LAYER 2 ==========")
l2_results, robust_docs = layer2_defend(surviving)

print("\n--- Layer 2 Per Doc ---")
for r in l2_results:
    status = "DROPPED" if r["dropped"] else "PASSED"
    print(f"  Doc {r['doc_index']} [{status}]")
    print(f"    Snippet     : {r['doc_snippet']}")
    print(f"    Provenance  : {r['provenance_weight']}")
    print(f"    Claims      : {r['claims_passing']}/{r['claims_checked']} passing (ratio: {r['pass_ratio']})")

print(f"\nFinal survivors after both layers: {len(robust_docs)} of {len(docs)}")
print("\n--- Final clean docs ---")
for doc in robust_docs:
    print(f"  → {doc}")

========== LAYER 1 ==========
Running Stage 1: QPC...
Query variants: ['What are the long-term health effects of excessive sugar consumption?', 'How long are the effects of excess sugar consumption on health?', 'How long are the effects of excess sugar consumption on your health?', 'How long are the effects of excess sugar consumption on the health?']
Running Stage 2: Pseudo-Query Inversion...

--- Layer 1 Per Doc ---
  Doc 0 [PASSED] → Excessive sugar intake is associated with obesity, insulin resistance, and an in...
  Doc 1 [PASSED] → Studies show that high sugar consumption contributes to cardiovascular diseases ...
  Doc 2 [PASSED] → Frequent intake of sugary beverages can negatively affect liver function and inc...
  Doc 3 [PASSED] → Research from multiple medical organizations indicates that reducing added sugar...
  Doc 4 [DROPPED] → Top 10 dessert recipes for family gatherings and celebrations....
  Doc 5 [DROPPED] → Buy organic honey and sweeteners online at discounted prices